In this notebook, I classify the videos into categories based on their title.

Create the connection to the database.

In [2]:
import sqlite3
import pandas as pd 

conn = sqlite3.connect('../data/lafc_content.db')

Open the stored SQL query with combined video vs lafc match context table, and pull it into the base_df dataframe.

In [3]:
query = '''
SELECT *
FROM videos 
'''
videos_df = pd.read_sql(query, conn)

videos_df

,video_id,channel_id,title,description,published_at,duration,view_count,like_count,comment_count,fetched_at
0,J8KtvKKDsBI,UCnqj91wjXAT0bsmxF1fHWBA,Inside LAFC | Episode 211 - A Strong Start,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-07-21T08:27:48Z,PT49M46S,1129,88,4,2026-07-21T18:13:52.868219+00:00
1,6IGiJLX6zIA,UCnqj91wjXAT0bsmxF1fHWBA,Sonny's goal from pitchside 🤳,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-07-21T05:15:15Z,PT17S,9169,901,20,2026-07-21T18:13:52.868219+00:00
2,dk5FTY2zHEI,UCnqj91wjXAT0bsmxF1fHWBA,Son Heung-Min | EVERY ANGLE of his derby goal ...,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-07-20T07:17:45Z,PT2M10S,10782,1373,100,2026-07-21T18:13:52.868219+00:00
3,WGLkpecuCyA,UCnqj91wjXAT0bsmxF1fHWBA,LAFC Weekly | Episode 15 | 2026,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-07-19T01:00:21Z,PT21M,2106,167,9,2026-07-21T18:13:52.868219+00:00
4,rjdbdSE3TNo,UCnqj91wjXAT0bsmxF1fHWBA,A Night To Remember | LAG vs LAFC,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-07-18T22:58:28Z,PT25S,2759,457,31,2026-07-21T18:13:52.868219+00:00
...,...,...,...,...,...,...,...,...,...,...
3566,8yoRN5MvJP0,UCnqj91wjXAT0bsmxF1fHWBA,Somos LAFC,"Nuestra Ciudad, Nuestro Club, Nuestro Escudo. ...",2016-01-08T00:27:56Z,PT2M,10264,165,16,2026-07-21T18:13:52.868219+00:00
3567,Emczin-vSFQ,UCnqj91wjXAT0bsmxF1fHWBA,WE ARE LAFC,"Our City, Our Club, Our Crest. We are LAFC.",2016-01-07T17:51:44Z,PT2M,146275,1160,191,2026-07-21T18:13:52.868219+00:00
3568,a63ytKgBTAc,UCnqj91wjXAT0bsmxF1fHWBA,John Thorrington announcement on SportsCenter,"Check out SportsCenter, giving some airtime to...",2015-12-09T23:54:13Z,PT27S,2094,24,2,2026-07-21T18:13:52.868219+00:00
3569,QJP0ITdmAeo,UCnqj91wjXAT0bsmxF1fHWBA,Building Together: LAFC Stadium Workshop,We asked our supporters to help us design our ...,2015-11-24T19:50:33Z,PT1M1S,6295,73,5,2026-07-21T18:13:52.868219+00:00


Import Tfid and kMeans from scikit learn.

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

Create a series of titles from the base data frame, I'm ignoring the description because it's repeated across different types of videos, and biases clustering.

In [4]:
text = videos_df['title'].fillna('')
text.head()

0           Inside LAFC | Episode 211 - A Strong Start
1                        Sonny's goal from pitchside 🤳
2    Son Heung-Min | EVERY ANGLE of his derby goal ...
3                      LAFC Weekly | Episode 15 | 2026
4                    A Night To Remember | LAG vs LAFC
Name: title, dtype: str

Adding (too) commonly occuring title words to the English stop words list:

In [5]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
custom = ENGLISH_STOP_WORDS.union({'lafc','https','com','www','la','los','angeles','ep','episode'})

Vectorize the text series, into a sparse matrix, using tfid, and load it in a feature variable X.

In [6]:
vec = TfidfVectorizer(
    stop_words=list(custom),   # use my custom words as stop_words
    ngram_range=(1, 2),     # single words AND 2-word phrases ("inside lafc")
    min_df=5,               # ignore terms that appear in fewer than 5 videos (rare noise)
    max_df=0.4,
    max_features=500,       # cap vocabulary size — keeps it fast/focused
)
X = vec.fit_transform(text)

Create the kmeans model and assign each video a cluster.

In [7]:
k = 8
km = KMeans(n_clusters=k, random_state=11, n_init=10)
videos_df['cluster'] = km.fit_predict(X)

Examining the clusters:

In [8]:
terms = vec.get_feature_names_out()
for c in range(k):
    top_idx = km.cluster_centers_[c].argsort()[-10:][::-1]   # 10 highest-weight terms
    keywords = ", ".join(terms[i] for i in top_idx)
    examples = videos_df[videos_df.cluster == c]['title'].head(3).tolist()
    print(f"\ncluster {c} (n={(videos_df.cluster==c).sum()})")
    print(f"  keywords: {keywords}")
    print(f"  examples: {examples}")


cluster 0 (n=2193)
  keywords: mls, weekly, football, bouanga, vs, crest, banc, armando, acción armando, acción
  examples: ['LAFC Weekly | Episode 15 | 2026', 'A Night To Remember | LAG vs LAFC', 'Sonny back on the scoresheet 🫡']

cluster 1 (n=126)
  keywords: media, vs, prematch media, prematch, postmatch media, postmatch, vs prematch, vs postmatch, col, atx
  examples: ['LAG vs LAFC | Postmatch Media', 'LAG vs LAFC | Prematch Media', 'LAFC vs SEA | Postmatch Media']

cluster 2 (n=82)
  keywords: mvp podcast, mvp, podcast, special, special guest, guest, cup, ilie, john, jordan harvey
  examples: ['MVP Podcast Ep. 77 - Time To Regroup with special guest Denil Maldonado', 'MVP Podcast Ep. 76 - Road Work with Special Guest Daniel Crisostomo', 'MVP Podcast Ep. 75 - Changing The Batteries with Special Guest Julian Gaines']

cluster 3 (n=163)
  keywords: inside, inside podcast, podcast, max vince, vince, max, inside max, cup, john, home
  examples: ['Inside LAFC | Episode 211 - A Strong S

Based on the clustered keywords, I created a rules based classify function. I also created a format family dictionary, to group related cateogries, post function.

In [4]:
def classify_format(title):
    """
    Rule-based format classifier for LAFC videos.
    Order matters: named series first, then press/interview, then match content,
    then themed content, catch-all last. First match wins. Title-only, no regex.
    """
    t = str(title).lower()

    # 1. Named recurring series
    if 'inside lafc' in t:                                          return 'inside_lafc'
    if 'gold insider' in t:                                         return 'black_and_gold'   # the SHOW only
    if 'is black & gold' in t or 'is black and gold' in t:          return 'signing'          # "X is Black & Gold"
    if 'mvp podcast' in t:                                          return 'mvp_podcast'
    if 'acción' in t or 'accion' in t:                             return 'accion_lafc'
    if 'lafc weekly' in t or 'weekly | ' in t:                     return 'lafc_weekly'
    if 'lafc+' in t or 'lafc +' in t:                              return 'lafc_plus'
    if 'on the mic' in t:                                           return 'on_the_mic'
    if 'behind the crest' in t:                                     return 'behind_the_crest'
    if 'away days' in t:                                            return 'away_days'
    if 'a lot more to prove' in t:                                  return 'more_to_prove'
    if 'staying home' in t:                                         return 'staying_home'
    if '안녕 lafc' in t:                                            return 'korean_series'
    if 'podcast' in t:                                              return 'other_podcast'

    # 2. Press / interview (keyword-based)
    if 'postmatch' in t or 'post-match' in t:                      return 'postmatch_media'
    if 'prematch' in t or 'pre-match' in t:                        return 'prematch_media'
    if 'conference' in t or 'presser' in t or 'media availab' in t:  # 'availab' catches the typo
        return 'presser'
    if any(w in t for w in ['in touch','speaks','discusses','talks ',
                            'thoughts','reacts','reaction','breaks down','sits down']):
        return 'interview'

    # 3. Match content
    if 'highlight' in t or 'all goals' in t or 'every goal' in t or 'goal scored' in t \
       or 'top ten goals' in t or 'top 10 goals' in t or 'best goals' in t or 'top goals' in t:
        return 'highlights'
    if ('goal:' in t or t.startswith('goal ') or t.startswith('goal!')
            or '| goal' in t or 'every angle' in t or 'wondergoal' in t or 'golazo' in t
            or 'game winner' in t or 'from the spot' in t or 'from pitchside' in t):
        return 'goal_clip'
    if 'save of the match' in t or 'huge save' in t or 'leaping save' in t or 'big save' in t:
        return 'save_clip'
    if 'match frames' in t:                                         return 'match_frames'
    if 'preview' in t or 'keys to the match' in t:                 return 'match_preview'
    if 'recap' in t:                                                return 'recap'

    # 4. Themed / behind-the-scenes / features
    if 'behind the scenes' in t or 'sounds of' in t:               return 'behind_the_scenes'
    if 'training' in t or "mic'd up" in t or 'micd up' in t:       return 'training'
    if 'watch party' in t or 'watch along' in t or 'watch-along' in t or '110 football' in t:
        return 'watch_party'
    if 'built for it' in t:                                         return 'built_for_it'
    if any(w in t for w in ['get to know','player profile','lafc profile','join the club',
                            'on this day','cali to cali']):         return 'feature'

    # 5. Catch-all
    return 'unclassified'


FORMAT_FAMILY = {
    # produced recurring series
    'inside_lafc':      'show',
    'black_and_gold':   'show',
    'mvp_podcast':      'show',
    'accion_lafc':      'show',
    'lafc_weekly':      'show',
    'lafc_plus':        'show',
    'on_the_mic':       'show',
    'behind_the_crest': 'show',
    'away_days':        'show',
    'more_to_prove':    'show',
    'staying_home':     'show',
    'korean_series':    'show',
    'other_podcast':    'show',

    # match content
    'highlights':       'match',
    'goal_clip':        'match',
    'save_clip':        'match',
    'match_frames':     'match',
    'match_preview':    'match',
    'recap':            'match',

    # press / interview
    'postmatch_media':  'media',
    'prematch_media':   'media',
    'presser':          'media',
    'interview':        'media',

    # themed / features
    'behind_the_scenes':'behind_scenes',
    'training':         'behind_scenes',
    'watch_party':      'watch_party',
    'built_for_it':     'feature',
    'feature':          'feature',
    'signing':          'signing',

    # catch-all
    'unclassified':     'unclassified', 
}

Apply the classify function, and map the format family, to two new columns in the base_df, then count them:

In [5]:
videos_df['content_type']   = videos_df['title'].apply(classify_format)
videos_df['format_family']  = videos_df['content_type'].map(FORMAT_FAMILY)

videos_df['content_type'].value_counts()

content_type
unclassified         1903
highlights            247
goal_clip             169
inside_lafc           164
lafc_weekly           111
lafc_plus             103
accion_lafc            86
mvp_podcast            82
recap                  65
match_preview          64
signing                63
prematch_media         61
postmatch_media        57
black_and_gold         55
interview              55
behind_the_crest       49
feature                47
presser                40
watch_party            40
training               36
behind_the_scenes      21
korean_series          12
match_frames           11
away_days               9
on_the_mic              8
save_clip               4
other_podcast           3
staying_home            3
built_for_it            2
more_to_prove           1
Name: count, dtype: int64

Even with all my rules, many of the videos are not easily classifiable by title, those are assigned the category of "unclassified".

In [11]:
videos_df[['title', 'format_family']].sample(30)

,title,format_family
852,Xolo vibe check ✔️,unclassified
1772,LAFC+ | Ep. 14,show
2693,LAFC Profile | Canada Connection Dejan Jakovic...,feature
129,Inside LAFC | Episode 203 - Looking ahead,show
3250,GOAL Walker Zimmerman | LAFC 1 - 0 COL,match
51,LAFC vs SEA | Postmatch Media,media
1445,First goal in Black & Gold for Lewis O'Brien |...,unclassified
126,Acción LAFC Con Armando Aguayo | Ep. 84,show
1874,"Part of our History | Diego ""Chiqui"" Palacios",unclassified
1736,LAFC Weekly Presented by Carmax - Ep. 9,show


I tuned the classification based on rules as best I could, but I still had ≈ 1800 videos that I was labeling "unclassified" they were mostly short social clips without a standard title, but some were interview clips, so I decided to try a light machine learning model on the "unclassified" set to see if I could label those more finely. 

In [6]:
from sentence_transformers import SentenceTransformer, util
model = SentenceTransformer('all-MiniLM-L6-v2')

/Users/laptop_02/Documents/data_projects/lafc_content/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
'[Errno 8] nodename nor servname provided, or not known' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json
Retrying in 1s [Retry 1/5].


RuntimeError: Cannot send a request, as the client has been closed.

In [13]:
CATEGORIES = {
    'goal_clip':   'a video clip of a single goal being scored in a match',
    'highlights':  'match highlights or a compilation of multiple goals',
    'interview':   'a player or coach speaking, quote, press interview or reaction',
    'feature':     'a player profile, personal story, or getting-to-know feature',
    'behind_the_scenes': 'behind the scenes footage, training, or documentary content',
    'match_preview':'a preview or build-up before an upcoming match',
    'recap':       'a season or match recap looking back',
    'misc_social_clip': 'a short hype or social media clip, slogan, or promotional post',
}

In [14]:
cat_names = list(CATEGORIES.keys())
cat_embeddings = model.encode(list(CATEGORIES.values()))

In [15]:
unclassified = videos_df[videos_df['content_type'] == 'unclassified'].copy()
title_embeddings = model.encode(unclassified['title'].fillna('').tolist(), show_progress_bar=True)

Batches: 100%|██████████| 60/60 [00:01<00:00, 32.20it/s]


In [16]:
import numpy as np
sims = util.cos_sim(title_embeddings, cat_embeddings).numpy()   # shape: (n_titles, n_categories)
best_idx   = sims.argmax(axis=1)     # which category scored highest per title
best_score = sims.max(axis=1)        # how strong that match was
unclassified['ml_label'] = [cat_names[i] for i in best_idx]
unclassified['ml_score'] = best_score

In [17]:
THRESHOLD = 0.30   # tune this — see below
unclassified['ml_label'] = np.where(
    unclassified['ml_score'] >= THRESHOLD,
    unclassified['ml_label'],
    'misc_social_clip'          # weak match -> genuine miscellany
)

In [18]:
unclassified[['title', 'ml_label', 'ml_score']].sample(30)

,title,ml_label,ml_score
3113,Zimmerman and Ramirez Return From USMNT Camp,misc_social_clip,0.241218
3254,Anatomy Of A Goal | Diego Rossi vs Chicago Fir...,goal_clip,0.512476
895,Nathan Ordaz goal vs HOU,highlights,0.323035
2467,What It Will Take To Conquer The CONCACAF Cham...,misc_social_clip,0.214680
2648,LAFC Welcomes New 2020 Members At Banc Of Cali...,misc_social_clip,0.109377
459,New friends 🤝,misc_social_clip,0.190019
788,Bouanga Magic 🪄,misc_social_clip,0.084287
1302,"Cherundolo: We're Ready, Have Some Ideas In Pl...",misc_social_clip,0.158120
2546,LAFC x Farmer John LA | Bresee Foundation #LAF...,misc_social_clip,0.147108
504,SJ vs LAFC | Every Assist,misc_social_clip,0.264993


In [19]:
unclassified['ml_score'].describe()

count    1903.000000
mean        0.248524
std         0.093207
min         0.024242
25%         0.182518
50%         0.244476
75%         0.307881
max         0.584862
Name: ml_score, dtype: float64

In [20]:
unclassified['dur_min'] = pd.to_timedelta(unclassified['duration']).dt.total_seconds() / 60

u = unclassified.sort_values('ml_score', ascending=False)

print("=== HIGH scores (top matches) ===")
print(u.head(15)[['title','ml_label','ml_score','dur_min']].round({'ml_score':3,'dur_min':1}).to_string())

print("\n=== MID scores (around the median) ===")
print(u.iloc[900:915][['title','ml_label','ml_score','dur_min']].round({'ml_score':3,'dur_min':1}).to_string())

print("\n=== LOW scores (weakest) ===")
print(u.tail(15)[['title','ml_label','ml_score','dur_min']].round({'ml_score':3,'dur_min':1}).to_string())

=== HIGH scores (top matches) ===
                                                                         title    ml_label  ml_score  dur_min
3479                    WATCH: A closer look at the first goal in LAFC history   goal_clip     0.585      0.5
405                                             ✌️ Regular season matches left       recap     0.570      0.3
714                                    Goals in back-to-back matches for Ordaz  highlights     0.527      0.5
853                                                     4️⃣ goals in one place  highlights     0.526      0.6
2882                       11 Goals Over In Our Last 2 Games | Watch Them All!  highlights     0.524      1.0
1446                    1 Goal = 3 Celebrations | #LAFC #MLS #LeaguesCup #Goal  highlights     0.518      0.2
2917                Anatomy Of A Goal | Diego Rossi vs Portland Timbers 6/1/19   goal_clip     0.517      2.4
3106                                            The Can't Miss Matches Of 2019       r

The light ml model did well at recognizing goal and highlight clips, but it was missing interview clips. I decide to provide the model with category vectors based on actual examples titles for each category, instead of providing it with text descriptions of the categories, to see if that would improve its classification.

In [21]:
PROTOTYPES = {
    'goal_clip': [
        # structured (with score line)
        "GOAL: M. Bogusz vs VAN, 1'",
        "Denis Bouanga breaks the tie! LAFC 2 - 1 HOU",
        "Diego Rossi opens the scoring, LAFC 1 - 0 Dallas",
        # short descriptive goal moments (no score line — the leaking shape)
        "Sonny picks his spot 🎯",
        "Near post finish by Bouanga 😮‍💨",
        "Bouanga chips the keeper | ALL ANGLES",
        "SONNY FROM DISTANCE 🚀",
        "Timmy's strike from the top of the box",
    ],

    'highlights': [
        "Highlights | LAFC vs FC Dallas",
        "Full Highlights | 3-0 | LAFC vs. Colorado Rapids",
        "MATCH HIGHLIGHTS | LAFC vs Seattle Sounders",
        "Every Goal From LAFC's Inaugural MLS Season",
        "11 Goals Over In Our Last 2 Games | Watch Them All",
    ],
    'interview': [
        "Cherundolo: We'll Need Effort Again Against Austin",
        "Ebobisse: Feeling more and more confident by the day",
        "Bradley Addresses Media After Mark-Anthony Kaye Trade",
        "Bouanga speaks on his hat trick",
        "Nguyen: This Is Where You Start To Play For Playoff Positions",
        "State Of The Union | Tom Penn",
        "Hollingshead: Huge Result For Us, Three Points On The Road",
    ],
    'feature': [
        "Get To Know Kwadwo Opoku",
        "LAFC Profile | From Norway to LA, Adama Diomande",
        "Building A Legacy | Carlos Vela's Past & Future With LAFC",
        "Join The Club | Juan Pinto",
        "The Call-Up | Christian Ramirez",
    ],
    'behind_the_scenes': [
        "Behind The Scenes | 2026 Primary Kit Shoot",
        "Sounds of Training | First Week Back",
        "A Look Behind The Scenes With Equipment Manager Scott Tranilla",
        "Inside the locker room after the win",
    ],
    'match_preview': [
        "LAFC at LA Galaxy - Match Preview",
        "Keys To The Match | LAFC vs Seattle",
        "Previewing the road trip to Colorado",
        "What to watch for ahead of LAFC vs Austin FC",
    ],
    'recap': [
        "2023 LAFC Season Recap",
        "Recap | LAFC vs Colorado Rapids",
        "Looking Back At The 2022 MLS Cup Run",
        "Year In Review | 2021 Season",
    ],
    'misc_social_clip': [
        "24 Hours ⏳",
        "The dagger 🗡️",
        "99 is electric ⚡️",
        "Ready to fight for the Club 🫡",
        "Pressure is a privilege",
        "It's about the collective.",
    ],
}

In [22]:
import numpy as np

cat_names = list(PROTOTYPES.keys())
cat_vectors = []
for name in cat_names:
    ex_embs = model.encode(PROTOTYPES[name])   # embed that category's example titles
    cat_vectors.append(ex_embs.mean(axis=0))   # average -> one "centroid" per category
cat_vectors = np.vstack(cat_vectors)

In [23]:
from sentence_transformers import util

sims = util.cos_sim(title_embeddings, cat_vectors).numpy()
best_idx = sims.argmax(axis=1)
unclassified['ml_label'] = [cat_names[i] for i in best_idx]
unclassified['ml_score'] = sims.max(axis=1)

In [24]:
THRESHOLD = 0.47   # tuned based on running the model a few times at looking at resulting ml scores against titles
unclassified['ml_label'] = np.where(
    unclassified['ml_score'] >= THRESHOLD,
    unclassified['ml_label'],
    'misc_social_clip'
)

In [25]:
unclassified[['title', 'ml_label', 'ml_score']].sample(30)

,title,ml_label,ml_score
1760,Cherundolo: The Performance Was Professional A...,misc_social_clip,0.347951
1398,Segura: Segura: Nos Llevamos Un Punto Bastante...,misc_social_clip,0.269492
1546,Cherundolo: We Saw Opportunities In Transition...,interview,0.497006
606,Son Heung-Min Makes His LAFC Debut.,misc_social_clip,0.455461
788,Bouanga Magic 🪄,goal_clip,0.485454
2658,El Capitán | What Motivates Carlos Vela,feature,0.479225
848,"No boot, no problem 🤷‍♂️",misc_social_clip,0.162450
2467,What It Will Take To Conquer The CONCACAF Cham...,misc_social_clip,0.389635
526,"Good morning, LA 🤩",misc_social_clip,0.380172
2745,LAFC x Porsche | A Look Inside Lee Nguyen's Pr...,misc_social_clip,0.424011


In [26]:
unclassified['ml_score'].describe()

count    1903.000000
mean        0.442647
std         0.110776
min         0.122112
25%         0.366663
50%         0.443994
75%         0.514534
max         0.823427
Name: ml_score, dtype: float64

In [27]:
unclassified['dur_min'] = pd.to_timedelta(unclassified['duration']).dt.total_seconds() / 60

u = unclassified.sort_values('ml_score', ascending=False)

print("=== HIGH scores (top matches) ===")
print(u.head(15)[['title','ml_label','ml_score','dur_min']].round({'ml_score':3,'dur_min':1}).to_string())

print("\n=== MID scores (around the median) ===")
print(u.iloc[900:915][['title','ml_label','ml_score','dur_min']].round({'ml_score':3,'dur_min':1}).to_string())

print("\n=== LOW scores (weakest) ===")
print(u.tail(15)[['title','ml_label','ml_score','dur_min']].round({'ml_score':3,'dur_min':1}).to_string())

=== HIGH scores (top matches) ===
                                                                                       title       ml_label  ml_score  dur_min
310                                                    A Year In Review | LAFC's 2025 Season          recap     0.823      4.6
3152                                             MEMORABLE MATCHDAY | LAFC Makes MLS History     highlights     0.768      3.9
2594                                               LAFC In 30 | LAFC vs. FC Dallas - 5/19/19     highlights     0.737     30.0
3454                                     WATCH: All 3 Goals in LAFC's 3-4 Loss vs. LA Galaxy     highlights     0.736      2.2
3442                                 WHAT A MATCH: LAFC vs. Montreal Impact | April 21, 2018     highlights     0.735      4.0
3479                                  WATCH: A closer look at the first goal in LAFC history     highlights     0.733      0.5
1716                                                       2024 Preseason: LA

It did a better job at classifying, and I went back and adjusted the ml_score threshold (below that the model would default a row to misc_social_clip). Then I merged the data frame of previously unclassified, back into the base_df, adding a content_type_final_column.

In [1]:
import numpy as np

# 1. Apply the threshold -> final label for the unclassified rows
THRESHOLD = 0.47
unclassified['final_label'] = np.where(
    unclassified['ml_score'] >= THRESHOLD,
    unclassified['ml_label'],
    'misc_social_clip'
)

# FORMAT_FAMILY was written before the ML pass existed, so it has no entry
# for the one category only the model can produce. Add it here rather than
# editing the original dict, to keep the order of discovery visible.
FORMAT_FAMILY['misc_social_clip'] = 'social'

# 2. Start the merged column as the RULE labels (the trustworthy core)
videos_df['content_type_final'] = videos_df['content_type']

# 3. Overwrite ONLY the previously-unclassified rows with the ML result.
#    Aligns by index — works because `unclassified` kept base_df's index.
videos_df.loc[unclassified.index, 'content_type_final'] = unclassified['final_label']

# 4. Re-map the coarse family on the merged labels
videos_df['format_family'] = videos_df['content_type_final'].map(FORMAT_FAMILY)

# Add the duration minutes column from the unclassified df
videos_df['dur_min'] = unclassified['dur_min']

# 5. Sanity checks
print(videos_df['content_type_final'].value_counts(), "\n")
print("misc_social_clip:", f"{(videos_df.content_type_final=='misc_social_clip').mean():.0%}")
print("unmapped families (must be 0):", videos_df['format_family'].isna().sum())

NameError: name 'unclassified' is not defined

In [30]:
print(videos_df['format_family'].value_counts(), '\n')

format_family
social           1193
match            1138
show              686
media             269
feature           125
signing            63
behind_scenes      57
watch_party        40
Name: count, dtype: int64 



In [31]:
videos_df

,video_id,channel_id,title,description,published_at,duration,view_count,like_count,comment_count,fetched_at,cluster,content_type,format_family,content_type_final
0,J8KtvKKDsBI,UCnqj91wjXAT0bsmxF1fHWBA,Inside LAFC | Episode 211 - A Strong Start,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-07-21T08:27:48Z,PT49M46S,1129,88,4,2026-07-21T18:13:52.868219+00:00,3,inside_lafc,show,inside_lafc
1,6IGiJLX6zIA,UCnqj91wjXAT0bsmxF1fHWBA,Sonny's goal from pitchside 🤳,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-07-21T05:15:15Z,PT17S,9169,901,20,2026-07-21T18:13:52.868219+00:00,5,goal_clip,match,goal_clip
2,dk5FTY2zHEI,UCnqj91wjXAT0bsmxF1fHWBA,Son Heung-Min | EVERY ANGLE of his derby goal ...,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-07-20T07:17:45Z,PT2M10S,10782,1373,100,2026-07-21T18:13:52.868219+00:00,5,goal_clip,match,goal_clip
3,WGLkpecuCyA,UCnqj91wjXAT0bsmxF1fHWBA,LAFC Weekly | Episode 15 | 2026,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-07-19T01:00:21Z,PT21M,2106,167,9,2026-07-21T18:13:52.868219+00:00,0,lafc_weekly,show,lafc_weekly
4,rjdbdSE3TNo,UCnqj91wjXAT0bsmxF1fHWBA,A Night To Remember | LAG vs LAFC,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-07-18T22:58:28Z,PT25S,2759,457,31,2026-07-21T18:13:52.868219+00:00,0,unclassified,match,match_preview
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3566,8yoRN5MvJP0,UCnqj91wjXAT0bsmxF1fHWBA,Somos LAFC,"Nuestra Ciudad, Nuestro Club, Nuestro Escudo. ...",2016-01-08T00:27:56Z,PT2M,10264,165,16,2026-07-21T18:13:52.868219+00:00,0,unclassified,social,misc_social_clip
3567,Emczin-vSFQ,UCnqj91wjXAT0bsmxF1fHWBA,WE ARE LAFC,"Our City, Our Club, Our Crest. We are LAFC.",2016-01-07T17:51:44Z,PT2M,146275,1160,191,2026-07-21T18:13:52.868219+00:00,0,unclassified,match,highlights
3568,a63ytKgBTAc,UCnqj91wjXAT0bsmxF1fHWBA,John Thorrington announcement on SportsCenter,"Check out SportsCenter, giving some airtime to...",2015-12-09T23:54:13Z,PT27S,2094,24,2,2026-07-21T18:13:52.868219+00:00,0,unclassified,social,misc_social_clip
3569,QJP0ITdmAeo,UCnqj91wjXAT0bsmxF1fHWBA,Building Together: LAFC Stadium Workshop,We asked our supporters to help us design our ...,2015-11-24T19:50:33Z,PT1M1S,6295,73,5,2026-07-21T18:13:52.868219+00:00,0,unclassified,match,match_preview


Dropping the columns I don't need and creating a new data frame, so it won't later get confused with the videos sql table.

In [35]:
classified_videos_df = videos_df.drop(['cluster', 'content_type'], axis=1)
classified_videos_df

,video_id,channel_id,title,description,published_at,duration,view_count,like_count,comment_count,fetched_at,format_family,content_type_final
0,J8KtvKKDsBI,UCnqj91wjXAT0bsmxF1fHWBA,Inside LAFC | Episode 211 - A Strong Start,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-07-21T08:27:48Z,PT49M46S,1129,88,4,2026-07-21T18:13:52.868219+00:00,show,inside_lafc
1,6IGiJLX6zIA,UCnqj91wjXAT0bsmxF1fHWBA,Sonny's goal from pitchside 🤳,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-07-21T05:15:15Z,PT17S,9169,901,20,2026-07-21T18:13:52.868219+00:00,match,goal_clip
2,dk5FTY2zHEI,UCnqj91wjXAT0bsmxF1fHWBA,Son Heung-Min | EVERY ANGLE of his derby goal ...,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-07-20T07:17:45Z,PT2M10S,10782,1373,100,2026-07-21T18:13:52.868219+00:00,match,goal_clip
3,WGLkpecuCyA,UCnqj91wjXAT0bsmxF1fHWBA,LAFC Weekly | Episode 15 | 2026,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-07-19T01:00:21Z,PT21M,2106,167,9,2026-07-21T18:13:52.868219+00:00,show,lafc_weekly
4,rjdbdSE3TNo,UCnqj91wjXAT0bsmxF1fHWBA,A Night To Remember | LAG vs LAFC,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-07-18T22:58:28Z,PT25S,2759,457,31,2026-07-21T18:13:52.868219+00:00,match,match_preview
...,...,...,...,...,...,...,...,...,...,...,...,...
3566,8yoRN5MvJP0,UCnqj91wjXAT0bsmxF1fHWBA,Somos LAFC,"Nuestra Ciudad, Nuestro Club, Nuestro Escudo. ...",2016-01-08T00:27:56Z,PT2M,10264,165,16,2026-07-21T18:13:52.868219+00:00,social,misc_social_clip
3567,Emczin-vSFQ,UCnqj91wjXAT0bsmxF1fHWBA,WE ARE LAFC,"Our City, Our Club, Our Crest. We are LAFC.",2016-01-07T17:51:44Z,PT2M,146275,1160,191,2026-07-21T18:13:52.868219+00:00,match,highlights
3568,a63ytKgBTAc,UCnqj91wjXAT0bsmxF1fHWBA,John Thorrington announcement on SportsCenter,"Check out SportsCenter, giving some airtime to...",2015-12-09T23:54:13Z,PT27S,2094,24,2,2026-07-21T18:13:52.868219+00:00,social,misc_social_clip
3569,QJP0ITdmAeo,UCnqj91wjXAT0bsmxF1fHWBA,Building Together: LAFC Stadium Workshop,We asked our supporters to help us design our ...,2015-11-24T19:50:33Z,PT1M1S,6295,73,5,2026-07-21T18:13:52.868219+00:00,match,match_preview


Writing the classfied videos data frame to the sqlite database.

In [36]:
classified_videos_df.to_sql('classified_videos', conn, if_exists='replace', index=False)

3571

Testing if the new table is in the database and it is.

In [38]:
query = '''
SELECT *
FROM classified_videos 
'''
test = pd.read_sql(query, conn)

test

,video_id,channel_id,title,description,published_at,duration,view_count,like_count,comment_count,fetched_at,format_family,content_type_final
0,J8KtvKKDsBI,UCnqj91wjXAT0bsmxF1fHWBA,Inside LAFC | Episode 211 - A Strong Start,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-07-21T08:27:48Z,PT49M46S,1129,88,4,2026-07-21T18:13:52.868219+00:00,show,inside_lafc
1,6IGiJLX6zIA,UCnqj91wjXAT0bsmxF1fHWBA,Sonny's goal from pitchside 🤳,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-07-21T05:15:15Z,PT17S,9169,901,20,2026-07-21T18:13:52.868219+00:00,match,goal_clip
2,dk5FTY2zHEI,UCnqj91wjXAT0bsmxF1fHWBA,Son Heung-Min | EVERY ANGLE of his derby goal ...,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-07-20T07:17:45Z,PT2M10S,10782,1373,100,2026-07-21T18:13:52.868219+00:00,match,goal_clip
3,WGLkpecuCyA,UCnqj91wjXAT0bsmxF1fHWBA,LAFC Weekly | Episode 15 | 2026,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-07-19T01:00:21Z,PT21M,2106,167,9,2026-07-21T18:13:52.868219+00:00,show,lafc_weekly
4,rjdbdSE3TNo,UCnqj91wjXAT0bsmxF1fHWBA,A Night To Remember | LAG vs LAFC,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-07-18T22:58:28Z,PT25S,2759,457,31,2026-07-21T18:13:52.868219+00:00,match,match_preview
...,...,...,...,...,...,...,...,...,...,...,...,...
3566,8yoRN5MvJP0,UCnqj91wjXAT0bsmxF1fHWBA,Somos LAFC,"Nuestra Ciudad, Nuestro Club, Nuestro Escudo. ...",2016-01-08T00:27:56Z,PT2M,10264,165,16,2026-07-21T18:13:52.868219+00:00,social,misc_social_clip
3567,Emczin-vSFQ,UCnqj91wjXAT0bsmxF1fHWBA,WE ARE LAFC,"Our City, Our Club, Our Crest. We are LAFC.",2016-01-07T17:51:44Z,PT2M,146275,1160,191,2026-07-21T18:13:52.868219+00:00,match,highlights
3568,a63ytKgBTAc,UCnqj91wjXAT0bsmxF1fHWBA,John Thorrington announcement on SportsCenter,"Check out SportsCenter, giving some airtime to...",2015-12-09T23:54:13Z,PT27S,2094,24,2,2026-07-21T18:13:52.868219+00:00,social,misc_social_clip
3569,QJP0ITdmAeo,UCnqj91wjXAT0bsmxF1fHWBA,Building Together: LAFC Stadium Workshop,We asked our supporters to help us design our ...,2015-11-24T19:50:33Z,PT1M1S,6295,73,5,2026-07-21T18:13:52.868219+00:00,match,match_preview
